# Advanced sweep on a free GPU (Colab or Kaggle)

Runs groups from `configs/sweep_advanced.yaml` (see `docs/ADVANCED_EXPERIMENTS.md`).

**Before you start**
- Colab: *Runtime -> Change runtime type -> T4 GPU*. Kaggle: *Settings -> Accelerator -> GPU*, and turn *Internet* on.
- Kaggle is the better choice for long groups: 4 CPU cores (vs Colab's 2) and sessions are not killed for being idle.
- Runs are resumable. If the session dies, run the notebook again with the same `OUT`: finished runs are skipped.
- To split one group across several sessions, give each a different `SHARD` (`"0/3"`, `"1/3"`, `"2/3"`).

## 1. Get the code

In [ ]:
# If the repository is private, use https://<YOUR_GITHUB_TOKEN>@github.com/... instead.
!git clone -q -b claude/sleepy-faraday-8nd6j9 https://github.com/shizen-57/agentic_federated_learning.git
%cd agentic_federated_learning
!pip install -q pyyaml pyarrow pillow
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "no GPU -- runs will use the CPU"

## 2. Where results go

On Colab, saving to Google Drive keeps results if the session is reclaimed. On Kaggle, `/kaggle/working` is saved with the notebook version.

In [ ]:
import os
OUT = "results/adv"
if os.path.exists("/content"):            # Colab
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/afl_results/adv"
elif os.path.exists("/kaggle/working"):    # Kaggle
    OUT = "/kaggle/working/adv"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT)

## 3. Choose what to run

In [ ]:
GROUPS = "main_cifar"   # one name/pattern or comma-separated: "weak_cifar,main_femnist", "weak_*"
SHARD = "0/1"           # "i/n" to split across n sessions
JOBS = 4 if os.path.exists("/kaggle/working") else 2   # ~number of CPU cores
!python scripts/sweep.py --list --groups '{GROUPS}' --out {OUT} --shard {SHARD}

## 4. Data (FEMNIST needs a one-off conversion, ~1 minute)

In [ ]:
if "femnist" in GROUPS or "*" in GROUPS:
    !python scripts/prepare_femnist.py

## 5. Run

In [ ]:
!python scripts/sweep.py --groups '{GROUPS}' --out {OUT} --shard {SHARD} --jobs {JOBS} --set fl.device=auto

## 6. Summaries and paired significance tests

In [ ]:
import glob
for d in sorted(glob.glob(f"{OUT}/*/")):
    print("=" * 80, "\n", d)
    !python scripts/report.py --dir {d} --target 0.5
    !python scripts/stats.py --dir {d} --ref c2_rl_fedprox

## 7. Bring the results back

Zip the folder and download it, or copy it into `results/adv/` in your checkout and commit it (per-run CSVs are small and are meant to be committed).

In [ ]:
!cd {OUT}/.. && zip -qr adv_results.zip adv && ls -la adv_results.zip